In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import os
from langchain_openai import ChatOpenAI
from agents.sql_agent import *
from agents.orchestrator import *

In [33]:
max_sql_retries = 3
chat_history = 5
api_key_gpt = os.getenv("MY_API_KEY")

In [34]:
db_name = "finance.db"

db_description = """
Tables:

1. outcome
Columns:
- Date (DATE): expense date in YYYY-MM-DD format. There may be multiple expenses on the same day
- Value (FLOAT): amount of money spent (in polish zloty)
- Category (TEXT): type of expense. Options: czynsz, trip, services, yummy, restraunt, food, present, clothes, transport, outdoors, holiday, work, marta, projeb, health, electronics, house, cosmetics, education, charity, unknown, remont, jewellery
- Was planned (TEXT): whether the outcome was planned ('yes', 'no')
- Description (TEXT): a subcategory of category

2. income
Columns:
- Date (DATE): same as above
- Value (FLOAT): amount of money received
- Category (TEXT): type of income. Options: same as for outcome + salary, stypendium
- Constant (TEXT): whether the income is constant ('yes', 'no')
- Description (TEXT): same as above
"""

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI
api_key_gemini = os.getenv("GEMINI_KEY")
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key_gemini)

In [36]:
# model = ChatOpenAI(model="gpt-4", api_key=api_key_gpt)
sql_agent = SQLAgent(model, db_name, db_description, max_sql_retries)
buddy = DataBuddyAgent(model, sql_agent, chat_history)

In [ ]:
# "I want all expenses from zabka from 2025"

# "ile miesiecnie srednio trace na jedzenie"
# "What's the weather in London now?"
# "i want to see how many i spend in this year so far on clothes"


# "сколько денег уходит в среднем месяц на косметику c 2026 года"
# "сколько денег я в среднем трачу на косметику каждый месяц в этом году"
# "подожди, так я хочу чтобы было каждый месяц была сумма, и вот с этого средняя"

# "was i spending more than i year in 2025?"
# "Well, explan it"
# "But how high were my spendings and earnings in 2025?"

In [45]:
buddy.chat("сколько денег в среднем в месяц уходит на косметику в 2026 году")
# "сколько денег уходит в среднем месяц на косметику c 2026 года"

,AVG(monthly_total)
0,386.38


{'intent': 'CREATE_NEW_QUERY',
 'answer': 'Согласно предоставленным данным, в среднем в месяц на косметику в 2026 году уходит 386.38.\n\nЭтот результат был получен путем сначала суммирования всех расходов на косметику за каждый месяц 2026 года, а затем вычисления среднего значения этих ежемесячных сумм.',
 'last query': "SELECT AVG(monthly_total) FROM (SELECT SUM(Value) AS monthly_total FROM outcome WHERE Category = 'cosmetics' AND STRFTIME('%Y', Date) = '2026' GROUP BY STRFTIME('%Y-%m', Date)) AS monthly_expenses",
 'token spent': 2050}

In [44]:
buddy.end()

In [42]:
print(buddy)

User inputs: ['сколько денег уходит в среднем месяц на косметику c 2026 года']
Recognized intents: ['CREATE_NEW_QUERY']
Buddy answers: ['В среднем, с 2026 roku na kosmetyki wydaje się 386.38 pieniędzy miesięcznie.\n\nTen wynik uzyskano poprzez zsumowanie wszystkich wydatków na kosmetyki dla każdego miesiąca począwszy od stycznia 2026 roku, a następnie obliczenie średniej z tych miesięcznych sum.']
Generated queries: ["SELECT AVG(monthly_sum) FROM (SELECT SUM(Value) AS monthly_sum FROM outcome WHERE Category = 'cosmetics' AND Date >= '2026-01-01' GROUP BY strftime('%Y-%m', Date))"]
Refined user inputs: []
Total tokens spent: 1915


In [ ]:
a

In [ ]:
while True:
    user_input = input("Your question (or 'quit'): ")
    if user_input.lower() == "quit":
        break
    res = buddy.chat(user_input)
    print("\nAnswer:\n", res["answer"])
    print("Query:\n", res["last query"])
    print("Intent:\n", res["intent"])
    print("Tokens spent:\n", res["token spent"])

In [ ]:
print(buddy.queries[-1])

In [25]:
def query_executor(query, db="finance.db"):
    """Executes SQL query and returns a DataFrame. Returns error if invalid."""
    try:
        with sqlite3.connect(db) as conn:
            df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        return str(e)

In [39]:
query_executor(
    """
    SELECT AVG(monthly_value) 
    FROM (
        SELECT SUM(Value) AS monthly_value 
        FROM outcome 
        WHERE Category LIKE 'cosmetics' AND STRFTIME('%Y', Date) = STRFTIME('%Y', 'now') 
        GROUP BY STRFTIME('%Y-%m', Date)
    )
    """
)

,AVG(monthly_value)
0,386.38
